# **Get Current Members of the U.S. Congress**

> This notebook retrieves all current voting members of the U.S. Congress using the Congress.gov API, processes and cleans the data, removes non-voting territorial delegates, and saves a date-stamped CSV file.
It also includes an optional section for comparing snapshots between two dates to detect new and departed members.

In [1]:
# Import Libraries

import requests         # for API communication
import pandas as pd     # for data manipulation
import time             # for pacing API calls
import glob             # for locating snapshot files to compare later
import os


## **API Configuration**

Define the base endpoint for the 119th Congress, the pagination limit, and authentication headers with your Congress.gov API key.

In [2]:
BASE_URL = "https://api.congress.gov/v3/member/congress/119/"
LIMIT = 250
offset = 0

headers = {
    "X-API-Key": "76GTa7yKIRDorzISvdcSWbeSgZnc2HcGSK8UeIgZ"  # Replace with your valid key
}

all_members = []

> Get your own key at API Congress [Here](https://api.congress.gov/sign-up/).

## **Fetch Members with Pagination**

Congress.gov returns data in pages. The loop below:

1. Requests current members (currentMember=true)

2. Appends the JSON results to all_members

3. Stops when no more data is returned. A short delay avoids API rate-limit errors.

In [3]:
while True:
    params = {
        "currentMember": "true",
        "limit": LIMIT,
        "offset": offset
    }

    response = requests.get(BASE_URL, headers=headers, params=params)
    data = response.json()

    if "members" not in data:
        print("No more data or error in response.")
        break

    members = data["members"]
    if not members:
        break

    all_members.extend(members)
    print(f"Fetched {len(members)} members (offset {offset})")

    offset += LIMIT
    time.sleep(0.2)  # avoid rate-limiting

Fetched 250 members (offset 0)
Fetched 250 members (offset 250)
Fetched 36 members (offset 500)


## **Normalize JSON into a DataFrame**
Each member includes nested term information.
We flatten the data using `pandas.json_normalize`, making every term a separate row while preserving key metadata.

In [4]:
# Convert Json to DataFrame
df = pd.json_normalize(
    all_members,
    record_path=["terms", "item"],
    meta=[
        "bioguideId",
        "name",
        "state",
        "district",
        "partyName",
        "updateDate",
        #"url",
        "depiction" # Path to the nested imageUrl
    ],
    errors='ignore'
)

## **Extract Image URLs**
The field `depiction` contains profile images in a nested dictionary.
We extract `imageUrl` and drop the nested column.

In [5]:
df['imageUrl'] = df['depiction'].str.get('imageUrl').fillna('No Image Available')
df = df.drop(columns=['depiction'])

## **Filter Current Members Only**

Each member’s history includes all past terms.
We keep only the active term, identified by `endYear` being null.

In [6]:
df_current = df[df["endYear"].isnull()].reset_index(drop=True)

## **Remove Non-Voting Delegates**

Territorial delegates (e.g., Puerto Rico, D.C.) do not have full voting rights.
We remove them to keep only official voting members.

In [7]:
list_of_territories = [
    "American Samoa",
    "Guam",
    "Northern Mariana Islands",
    "Puerto Rico",
    "Virgin Islands",
    "District of Columbia"
]

non_voting_members = df_current[df_current["state"].isin(list_of_territories)]
non_voting_members

,chamber,startYear,endYear,bioguideId,name,state,district,partyName,updateDate,imageUrl
118,House of Representatives,1991,NaN,N000147,"Norton, Eleanor Holmes",District of Columbia,0,Democratic,2025-12-06T08:30:20Z,https://www.congress.gov/img/member/116_dg_dc_...
147,House of Representatives,2025,NaN,H001103,"Hernández, Pablo Jose",Puerto Rico,0,Democratic,2025-09-24T07:40:20Z,https://www.congress.gov/img/member/67742d980b...
170,House of Representatives,2025,NaN,K000404,"King-Hinds, Kimberlyn",Northern Mariana Islands,0,Republican,2025-09-24T07:40:19Z,https://www.congress.gov/img/member/67742f0a0b...
221,House of Representatives,2023,NaN,M001219,"Moylan, James C.",Guam,0,Republican,2025-09-24T07:40:18Z,https://www.congress.gov/img/member/m001219_20...
426,House of Representatives,2015,NaN,P000610,"Plaskett, Stacey E.",Virgin Islands,0,Democratic,2025-09-24T07:40:14Z,https://www.congress.gov/img/member/116_dg_vi_...
441,House of Representatives,2015,NaN,R000600,"Radewagen, Aumua Amata Coleman",American Samoa,0,Republican,2025-09-24T07:40:14Z,https://www.congress.gov/img/member/r000600_20...


## **Clean and Parse Names**

The API provides names in "Last, First M." format.
We extract first, middle, last, and nickname fields, plus a full formatted name.

In [8]:
import re

def parse_name(full_name):
    parts = full_name.split(", ")
    last_name = parts[0]
    first_middle = parts[1] if len(parts) > 1 else ""

    nickname = re.findall(r'"(.*?)"', first_middle)
    nickname = nickname[0] if nickname else None

    clean_first_middle = re.sub(r'"(.*?)"', '', first_middle).strip()
    parts2 = clean_first_middle.split(" ", 1)
    first_name = parts2[0]
    middle_name = parts2[1] if len(parts2) > 1 else None

    full_formatted = " ".join(filter(None, [first_name, middle_name, last_name]))

    return {
        "first_name": first_name,
        "middle_name": middle_name,
        "last_name": last_name,
        "nickname": nickname,
        "Name": full_formatted
    }

parsed = df_current["name"].apply(parse_name).apply(pd.Series)
df_current = pd.concat([df_current, parsed], axis=1)


## **Standardize Chamber Names**
Simplify the column values by shortening "House of Representatives" to "House" and keeping "Senate" unchanged.

In [9]:
df_current["Chamber"] = df_current["chamber"].replace(
    {"House of Representatives": "House", "Senate": "Senate"}
)

## **Select and Rename Relevant Columns**

Choose key columns and rename them for clarity and consistency with previous versions.

In [10]:
df_final = df_current[[
    "bioguideId", "Name", "first_name", "middle_name", "last_name",
    "nickname", "Chamber", "state", "district", "partyName",
    "startYear", "endYear", "updateDate", "imageUrl"
]].rename(columns={
    "bioguideId": "bioguide_id",
    "state": "State",
    "district": "District",
    "partyName": "Party",
    "startYear": "start_year",
    "endYear": "end_year",
    "updateDate": "update_date"
})

## **Save the Dataset**
We generate a date-stamped filename following this pattern:
`119th_Congress_YYYYMMDD.csv`

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# Define current date and Congress number
current_date = pd.Timestamp.today().strftime("%Y%m%d")
congress_number = 119

# Define the project root directory (Google Drive)
drive_root = "/content/drive/Shareddrives/Data Archive/Bridge_Grades"

# Define the relative output path from the project root
base_dir = os.path.join(
    drive_root,
    "Input_Files",
    "Snapshot_Raw_2026_06_02",
    "Raw_Others_2026_06_02"
)
# Check if the directory exists (do not create it)
if not os.path.isdir(base_dir):
    raise OSError(f"Directory does not exist: {base_dir}")

# Build the output file name and full path
output_filename = f"{congress_number}th_Congress_{current_date}.csv"
output_path = os.path.join(base_dir, output_filename)

# Save the final DataFrame to CSV
df_final.to_csv(output_path, index=False)

# Display confirmation messages
print(f"File saved as: {output_path}")
print(f"Total voting members: {len(df_final)}")

File saved as: /content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_Others_2026_06_02/119th_Congress_20260602.csv
Total voting members: 536
